# 03: Evaluation & Feedback Loops

This notebook covers automated evaluation of LLM outputs, LLM-as-a-Judge patterns, and building feedback loops for continuous improvement.

## Objectives
- Understand automated evaluation metrics for LLM outputs
- Implement LLM-as-a-Judge evaluation
- Build a simple evaluation pipeline
- Design feedback collection and analysis
- Explore A/B testing concepts for AI

## 1. Evaluation Metrics Overview

LLM evaluation falls into several categories:

| Category | Metrics | Use Case |
|----------|---------|----------|
| **Similarity** | ROUGE, BLEU, BERTScore | Text comparison with reference |
| **Quality** | Faithfulness, Relevance, Coherence | Output quality assessment |
| **Task-specific** | Accuracy, F1, Exact Match | Classification, QA |
| **Human-aligned** | Preference, Helpfulness | User satisfaction |

In [ ]:
import os
import time
import json
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key

## 2. LLM-as-a-Judge Concept

Using a more capable LLM to evaluate outputs from another LLM. This enables:
- Scalable evaluation without human annotators
- Consistent scoring criteria
- Detailed qualitative feedback

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Judge LLM (use a capable model)
judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Evaluation prompt template
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert evaluator. You must evaluate the response based on the criteria.
Return a JSON object with the following fields:
- score: integer from 1-5 (1=poor, 5=excellent)
- criteria_scores: object with individual criterion scores
- explanation: brief explanation of the score
- issues: list of any issues found

Return ONLY valid JSON, no other text."""),
    ("human", """Evaluate this response:

## Original Question
{question}

## Response to Evaluate
{response}

## Evaluation Criteria
- Relevance: Does it directly address the question?
- Accuracy: Is the information correct?
- Clarity: Is it easy to understand?
- Completeness: Does it cover the topic thoroughly?
- Conciseness: Is it appropriately brief without being incomplete?

Provide your evaluation as JSON.""")
])

# Create evaluation chain
judge_chain = judge_prompt | judge_llm | JsonOutputParser()

# Test with an example
result = judge_chain.invoke({
    "question": "What is a closure in Python?",
    "response": "A closure is a function that remembers the variables from its enclosing scope even after the outer function has finished executing."
})

print("=== Judge Evaluation ===")
print(json.dumps(result, indent=2))

## 3. Building an Evaluation Pipeline

In [ ]:
@dataclass
class EvaluationResult:
    """Result of evaluating a single response."""
    question: str
    response: str
    score: int
    criteria_scores: dict
    explanation: str
    issues: list[str]
    timestamp: float = field(default_factory=time.time)
    metadata: dict = field(default_factory=dict)

class EvaluationPipeline:
    """Automated evaluation pipeline for LLM outputs."""
    
    def __init__(self, judge_model: str = "gpt-4o"):
        self.judge_llm = ChatOpenAI(model=judge_model, temperature=0)
        self.results: list[EvaluationResult] = []
    
    def evaluate_single(self, question: str, response: str, 
                       criteria: Optional[list[str]] = None) -> EvaluationResult:
        """Evaluate a single question-response pair."""
        if criteria is None:
            criteria = ["Relevance", "Accuracy", "Clarity", "Completeness", "Conciseness"]
        
        criteria_text = "\n".join([f"- {c}: Rate 1-5" for c in criteria])
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert evaluator. Return JSON with: score (1-5), criteria_scores (object), explanation (string), issues (list)."),
            ("human", """Evaluate:
Question: {question}
Response: {response}

Criteria:
{criteria}

Return JSON.""")
        ])
        
        chain = prompt | self.judge_llm | JsonOutputParser()
        
        try:
            result = chain.invoke({
                "question": question,
                "response": response,
                "criteria": criteria_text
            })
            
            eval_result = EvaluationResult(
                question=question,
                response=response,
                score=result.get("score", 0),
                criteria_scores=result.get("criteria_scores", {}),
                explanation=result.get("explanation", ""),
                issues=result.get("issues", [])
            )
        except Exception as e:
            eval_result = EvaluationResult(
                question=question,
                response=response,
                score=0,
                criteria_scores={},
                explanation=f"Evaluation failed: {str(e)}",
                issues=[str(e)]
            )
        
        self.results.append(eval_result)
        return eval_result
    
    def evaluate_batch(self, qa_pairs: list[dict]) -> list[EvaluationResult]:
        """Evaluate a batch of question-response pairs."""
        results = []
        for pair in qa_pairs:
            result = self.evaluate_single(
                question=pair["question"],
                response=pair["response"]
            )
            results.append(result)
        return results
    
    def get_summary(self) -> dict:
        """Get summary statistics of all evaluations."""
        if not self.results:
            return {"error": "No evaluations yet"}
        
        scores = [r.score for r in self.results]
        all_issues = [issue for r in self.results for issue in r.issues]
        
        return {
            "total_evaluations": len(self.results),
            "average_score": sum(scores) / len(scores),
            "score_distribution": {i: scores.count(i) for i in range(1, 6)},
            "common_issues": self._get_common_issues(all_issues),
            "low_score_count": sum(1 for s in scores if s <= 2)
        }
    
    def _get_common_issues(self, issues: list[str]) -> list[dict]:
        """Find most common issues."""
        from collections import Counter
        # Simple keyword extraction
        keywords = []
        for issue in issues:
            words = issue.lower().split()
            keywords.extend([w for w in words if len(w) > 4])
        
        counts = Counter(keywords).most_common(5)
        return [{"keyword": k, "count": c} for k, c in counts]

In [ ]:
# Test the evaluation pipeline
pipeline = EvaluationPipeline()

# Sample QA pairs
test_data = [
    {
        "question": "What is recursion?",
        "response": "Recursion is when a function calls itself."
    },
    {
        "question": "What is recursion?",
        "response": "Recursion is a programming technique where a function calls itself directly or indirectly. Each recursive call works on a smaller subproblem until reaching a base case. For example, calculating factorial: factorial(n) = n * factorial(n-1), with factorial(0) = 1 as the base case. Without a base case, recursion would continue infinitely."
    },
    {
        "question": "Explain the difference between a list and a tuple in Python.",
        "response": "Lists use brackets and tuples use parentheses."
    }
]

# Run evaluation
print("=== Running Evaluations ===")
for i, pair in enumerate(test_data):
    result = pipeline.evaluate_single(pair["question"], pair["response"])
    print(f"\nEvaluation {i+1}:")
    print(f"  Score: {result.score}/5")
    print(f"  Explanation: {result.explanation[:100]}...")
    print(f"  Issues: {result.issues[:2]}...")

# Get summary
print("\n=== Evaluation Summary ===")
summary = pipeline.get_summary()
print(json.dumps(summary, indent=2))

## 4. Feedback Collection System

In [ ]:
@dataclass
class UserFeedback:
    """User feedback on a response."""
    response_id: str
    rating: int  # 1-5
    comment: Optional[str] = None
    tags: list[str] = field(default_factory=list)
    timestamp: float = field(default_factory=time.time)
    user_id: Optional[str] = None

class FeedbackCollector:
    """Collect and analyze user feedback."""
    
    def __init__(self):
        self.feedbacks: list[UserFeedback] = []
    
    def add_feedback(self, response_id: str, rating: int, 
                    comment: str = None, tags: list[str] = None) -> UserFeedback:
        """Add user feedback."""
        feedback = UserFeedback(
            response_id=response_id,
            rating=rating,
            comment=comment,
            tags=tags or []
        )
        self.feedbacks.append(feedback)
        return feedback
    
    def get_statistics(self) -> dict:
        """Get feedback statistics."""
        if not self.feedbacks:
            return {"error": "No feedback collected"}
        
        ratings = [f.rating for f in self.feedbacks]
        positive = sum(1 for r in ratings if r >= 4)
        negative = sum(1 for r in ratings if r <= 2)
        
        # Analyze tags
        all_tags = [tag for f in self.feedbacks for tag in f.tags]
        from collections import Counter
        tag_counts = Counter(all_tags).most_common(10)
        
        return {
            "total_feedback": len(self.feedbacks),
            "average_rating": sum(ratings) / len(ratings),
            "positive_percentage": positive / len(self.feedbacks) * 100,
            "negative_percentage": negative / len(self.feedbacks) * 100,
            "top_tags": tag_counts
        }
    
    def get_negative_feedback(self) -> list[UserFeedback]:
        """Get negative feedback for analysis."""
        return [f for f in self.feedbacks if f.rating <= 2]

# Example usage
collector = FeedbackCollector()

# Simulate feedback collection
sample_feedback = [
    ("resp_001", 5, "Very helpful explanation!", ["clear", "accurate"]),
    ("resp_002", 3, "Okay but could be more detailed", ["brief"]),
    ("resp_003", 1, "Incorrect information", ["inaccurate", "hallucination"]),
    ("resp_004", 4, "Good answer", ["accurate"]),
    ("resp_005", 2, "Not relevant to my question", ["irrelevant"]),
]

for resp_id, rating, comment, tags in sample_feedback:
    collector.add_feedback(resp_id, rating, comment, tags)

print("=== Feedback Statistics ===")
stats = collector.get_statistics()
print(json.dumps(stats, indent=2))

print("\n=== Negative Feedback ===")
negative = collector.get_negative_feedback()
for f in negative:
    print(f"  {f.response_id}: {f.comment}")

## 5. A/B Testing Concepts for AI

A/B testing in AI compares different versions of prompts, models, or pipelines.

In [ ]:
import random
from dataclasses import dataclass

@dataclass
class ABTestVariant:
    """A variant in an A/B test."""
    name: str
    config: dict
    impressions: int = 0
    conversions: int = 0
    scores: list[int] = field(default_factory=list)
    
    @property
    def conversion_rate(self) -> float:
        return self.conversions / self.impressions if self.impressions > 0 else 0
    
    @property
    def average_score(self) -> float:
        return sum(self.scores) / len(self.scores) if self.scores else 0

class ABTest:
    """A/B testing framework for AI outputs."""
    
    def __init__(self, name: str):
        self.name = name
        self.variants: list[ABTestVariant] = []
        self.results: list[dict] = []
    
    def add_variant(self, name: str, config: dict) -> ABTestVariant:
        variant = ABTestVariant(name=name, config=config)
        self.variants.append(variant)
        return variant
    
    def assign_variant(self) -> ABTestVariant:
        """Randomly assign a variant (uniform distribution)."""
        return random.choice(self.variants)
    
    def record_result(self, variant: ABTestVariant, score: int, 
                     converted: bool = False):
        """Record a result for a variant."""
        variant.impressions += 1
        variant.scores.append(score)
        if converted:
            variant.conversions += 1
    
    def get_results(self) -> dict:
        """Get test results."""
        results = {}
        for v in self.variants:
            results[v.name] = {
                "impressions": v.impressions,
                "avg_score": v.average_score,
                "conversion_rate": v.conversion_rate
            }
        return results
    
    def get_winner(self, metric: str = "avg_score") -> Optional[ABTestVariant]:
        """Determine the winning variant."""
        if not self.variants:
            return None
        
        if metric == "avg_score":
            return max(self.variants, key=lambda v: v.average_score)
        elif metric == "conversion_rate":
            return max(self.variants, key=lambda v: v.conversion_rate)
        return None

# Example: Testing two different prompts
test = ABTest("prompt-optimization-test")

# Add variants
variant_a = test.add_variant("concise-prompt", {
    "system": "Answer briefly."
})

variant_b = test.add_variant("detailed-prompt", {
    "system": "Provide detailed explanations with examples."
})

# Simulate test results
for _ in range(50):
    # Variant A: concise, might get lower scores but higher engagement
    v = test.assign_variant()
    score = random.randint(3, 5) if v.name == "concise-prompt" else random.randint(2, 5)
    converted = random.random() < (0.6 if v.name == "concise-prompt" else 0.4)
    test.record_result(v, score, converted)

print("=== A/B Test Results ===")
results = test.get_results()
for name, data in results.items():
    print(f"\n{name}:")
    print(f"  Impressions: {data['impressions']}")
    print(f"  Avg Score: {data['avg_score']:.2f}")
    print(f"  Conversion Rate: {data['conversion_rate']:.1%}")

winner = test.get_winner()
print(f"\nWinner: {winner.name}")

## 6. Continuous Improvement Loop

In [ ]:
class ImprovementLoop:
    """Automated improvement loop based on evaluations and feedback."""
    
    def __init__(self):
        self.evaluation_history: list[dict] = []
        self.improvement_actions: list[dict] = []
    
    def analyze_trends(self, evaluations: list[EvaluationResult]) -> dict:
        """Analyze evaluation trends."""
        if not evaluations:
            return {"error": "No evaluations"}
        
        scores = [e.score for e in evaluations]
        
        # Find common criteria weaknesses
        criteria_averages = {}
        for e in evaluations:
            for criterion, score in e.criteria_scores.items():
                if criterion not in criteria_averages:
                    criteria_averages[criterion] = []
                criteria_averages[criterion].append(score)
        
        weak_criteria = []
        for criterion, scores_list in criteria_averages.items():
            avg = sum(scores_list) / len(scores_list)
            if avg < 3.5:
                weak_criteria.append({"criterion": criterion, "avg_score": avg})
        
        return {
            "total_evaluations": len(evaluations),
            "overall_avg_score": sum(scores) / len(scores),
            "weak_criteria": sorted(weak_criteria, key=lambda x: x["avg_score"]),
            "low_score_percentage": sum(1 for s in scores if s <= 2) / len(scores) * 100
        }
    
    def suggest_improvements(self, analysis: dict) -> list[str]:
        """Suggest improvements based on analysis."""
        suggestions = []
        
        for criterion in analysis.get("weak_criteria", []):
            name = criterion["criterion"]
            if name == "Accuracy":
                suggestions.append("Consider using RAG to ground responses in source documents")
            elif name == "Relevance":
                suggestions.append("Review prompt templates for better task specification")
            elif name == "Completeness":
                suggestions.append("Add structured output requirements to prompts")
            elif name == "Clarity":
                suggestions.append("Simplify prompts or use a more capable model")
        
        if analysis.get("low_score_percentage", 0) > 20:
            suggestions.append("High failure rate detected - review model selection and prompts")
        
        return suggestions

# Example usage
loop = ImprovementLoop()

# Simulate analysis
analysis = {
    "total_evaluations": 100,
    "overall_avg_score": 3.2,
    "weak_criteria": [
        {"criterion": "Accuracy", "avg_score": 2.8},
        {"criterion": "Completeness", "avg_score": 3.1}
    ],
    "low_score_percentage": 25
}

suggestions = loop.suggest_improvements(analysis)
print("=== Improvement Suggestions ===")
for i, s in enumerate(suggestions, 1):
    print(f"{i}. {s}")

## 7. Practical Exercise

### Task
1. Build a RAG evaluation pipeline that:
   - Evaluates answer quality
   - Evaluates retrieval quality (are relevant documents retrieved?)
   - Tracks hallucination rate

2. Design a feedback loop that:
   - Collects user ratings
   - Identifies common failure patterns
   - Suggests prompt improvements

In [ ]:
# Your code here
# Build a RAG evaluation pipeline

## Summary

In this notebook, you learned:

1. **LLM-as-a-Judge** — Using LLMs to evaluate other LLMs at scale
2. **Evaluation Pipeline** — Building automated evaluation systems
3. **Feedback Collection** — Gathering and analyzing user feedback
4. **A/B Testing** — Comparing different approaches
5. **Continuous Improvement** — Using data to drive improvements

### Key Takeaways
- Automated evaluation enables scalable quality monitoring
- LLM-as-a-Judge provides consistent, detailed feedback
- User feedback complements automated metrics
- A/B testing helps optimize prompts and models
- Continuous improvement loops drive long-term quality

### Next Steps
- Apply these patterns to your own AI applications
- Set up production monitoring with LangSmith
- Explore advanced evaluation techniques